In [1]:
#234567890#234567890#234567890#234567890#234567890#234567890#234567890#23456789
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
BATCH = 64

### Working with Data

In [3]:
training_data = datasets.FashionMNIST(
    root='data', train=True, download=True, transform=ToTensor())
test_data = datasets.FashionMNIST(
    root='data', train=False, download=True, transform=ToTensor())

In [4]:
train_dataloader = DataLoader(training_data, batch_size=BATCH)
test_dataloader = DataLoader(test_data, batch_size=BATCH)

for X, y in test_dataloader:
    print(f'X.shape: {X.shape} (n, c, h, w)')
    print(f'y.shape: {y.shape} ({y.dtype})')
    break

X.shape: torch.Size([64, 1, 28, 28]) (n, c, h, w)
y.shape: torch.Size([64]) (torch.int64)


### Creating Models

In [5]:
device = (
    torch.accelerator.current_accelerator().type 
    if torch.accelerator.is_available() 
    else 'cpu')
print('Device:', device)

Device: mps


In [7]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10))

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stac(x)
        return logits

In [8]:
mod = NeuralNetwork().to(device)
print(mod)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [10]:
loss_f = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(mod.parameters(), lr=1e-3)

In [11]:
def train(loader, mod, loss_f, opt):
    size = len(loader.dataset)
    mod.train()
    for batch, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        preds = mod(X)
        loss = loss_f(preds, y)
        loss.backward()
        opt.step()
        opr.zero_grad()
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f'loss: {loss:>7f} ({current:>5d}/{size:>5d})')

In [ ]:
def test(...)